# Voice Symptom Intake & Documentation Assistant - Google Colab Deployment

This notebook deploys the Voice Symptom Intake & Documentation Assistant on Google Colab with GPU support.

## Advantages of Colab Deployment:
- **Free GPU Access** (Tesla T4 with 16GB VRAM)
- **No Local Setup Issues** (no Windows file locking, FFmpeg, etc.)
- **Faster Inference** (MedASR & MedGemma both run on GPU)
- **Public URL Access** via ngrok

## Important Notes:
- Sessions last up to 12 hours (free tier)
- You'll need a **Hugging Face token** with "Read" access
- Accept model terms for `google/medasr` and `google/medgemma-1.5-4b-it`

## Latest Updates:
- **Clinical Glass React UI** — React 19 + TypeScript frontend, glassmorphism design system, 6 themes, SOAP review workflow (Approve/Edit/Reject per section), real-time waveform + live transcript, conversation panel, monitoring & HIPAA dashboards
- **HIPAA Audit Trail** — Comprehensive data access logging, encryption at rest (AES-256-GCM), data retention auto-purge
- **Monitoring & Observability** — Prometheus metrics, structured logging with correlation IDs, health dashboard with alerts
- **API Rate Limiting & Queue** — Per-user rate limits, inference queue with position tracking & estimated wait times
- **Medical NER** — Automatic extraction of Conditions & Medications using SciSpaCy
- **Progressive Web App** — Offline recording, install-to-homescreen, background sync
- **Real-time streaming transcription** via WebSocket (words appear as you speak!)
- Enhanced JSON parsing for reliable documentation generation
- Robust error recovery with intelligent fallback strategies
- Optimized generation parameters for medical documentation

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies

Run this cell to install all required packages. Uses `%%capture` to suppress verbose pip output.

In [ ]:
%%capture
# Install transformers from specific commit for MedASR support
!pip install git+https://github.com/huggingface/transformers.git@65dc261512cbdb1ee72b88ae5b222f2605aad8e5

# Core web framework
!pip install fastapi uvicorn[standard] python-multipart websockets httpx

# ML / audio
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install accelerate librosa soundfile noisereduce audioread
!pip install openai-whisper

# Config & persistence
!pip install pydantic pydantic-settings python-dotenv
!pip install sqlalchemy[asyncio] aiosqlite

# Auth & security
!pip install passlib[bcrypt] python-jose[cryptography] pyotp

# Tunneling
!pip install pyngrok nest-asyncio

# Medical NER (SciSpaCy)
!pip install spacy scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# HIPAA Encryption at Rest
!pip install cryptography>=42.0.0

print("\u2705 All dependencies installed successfully!")

## Step 3: Upload Your Code

**Option A:** Upload project folder from your computer
- Click the folder icon on the left sidebar
- Upload the entire `voice-symptom-triage-assistant` folder

**Option B:** Clone from GitHub (if you've pushed your code)

In [ ]:
# Option B: Clone from GitHub (uncomment and modify if using)
# !git clone https://github.com/YOUR_USERNAME/voice-symptom-triage-assistant.git
# %cd voice-symptom-triage-assistant

# Option A: If uploaded manually, navigate to the folder
%cd /content/voice-symptom-triage-assistant

## Step 4: Configure Hugging Face Token & Environment

**Get your token from:** https://huggingface.co/settings/tokens

**Make sure you've accepted terms for:**
- https://huggingface.co/google/medasr
- https://huggingface.co/google/medgemma-1.5-4b-it

In [ ]:
import os

# REPLACE WITH YOUR HUGGING FACE TOKEN
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

# Create .env file with enhanced MedGemma parameters
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write("MEDASR_MODEL=google/medasr\n")
    f.write("MEDGEMMA_MODEL=google/medgemma-1.5-4b-it\n")
    f.write("DEVICE=cuda\n")
    f.write("ENABLE_GPU=true\n")

    # MedGemma Generation Parameters (optimized for JSON output)
    f.write("MEDGEMMA_MAX_TOKENS=1024\n")
    f.write("MEDGEMMA_REPETITION_PENALTY=1.1\n")

    # Audio settings
    f.write("AUDIO_SAMPLE_RATE=16000\n")
    f.write("MAX_AUDIO_DURATION_SECONDS=300\n")

    # Streaming transcription (GPU can handle 2s intervals)
    f.write("STREAMING_INTERVAL_SECONDS=2.0\n")

    # Rate Limiting & Inference Queue
    f.write("RATE_LIMITING_ENABLED=true\n")
    f.write("RATE_LIMIT_GENERAL_RPM=60\n")
    f.write("RATE_LIMIT_INFERENCE_RPM=10\n")
    f.write("QUEUE_MAX_CONCURRENT_INFERENCES=2\n")
    f.write("QUEUE_MAX_SIZE=20\n")
    f.write("QUEUE_TIMEOUT_SECONDS=120\n")
    f.write("QUEUE_ESTIMATED_INFERENCE_SECONDS=5\n")

    # Monitoring & Observability
    f.write("METRICS_ENABLED=true\n")
    f.write("STRUCTURED_LOGGING_ENABLED=true\n")
    f.write("METRICS_ENDPOINT_AUTH_REQUIRED=false\n")
    f.write("METRICS_ALERT_WINDOW_SECONDS=300\n")
    f.write("METRICS_ERROR_RATE_WARNING=0.1\n")
    f.write("METRICS_ERROR_RATE_CRITICAL=0.25\n")
    f.write("METRICS_LATENCY_WARNING_SECONDS=15\n")
    f.write("METRICS_LATENCY_CRITICAL_SECONDS=30\n")

    # HIPAA Encryption at Rest (disabled by default)
    f.write("ENCRYPTION_AT_REST_ENABLED=false\n")
    f.write("ENCRYPTION_MASTER_KEY=CHANGE_ME_IN_PRODUCTION\n")
    f.write("ENCRYPTION_KDF_ITERATIONS=100000\n")

    # Data Retention & Auto-Purge
    f.write("RETENTION_SESSIONS_DAYS=365\n")
    f.write("RETENTION_AUDIT_LOGS_DAYS=2555\n")
    f.write("AUTO_PURGE_ENABLED=false\n")
    f.write("AUTO_PURGE_INTERVAL_HOURS=24\n")

    # Model cache directory
    f.write("MODEL_CACHE_DIR=./models\n")
    f.write("LOG_LEVEL=INFO\n")

print("\u2705 Environment configured!")
print("   - Max Tokens: 1024 (complete documentation)")
print("   - Repetition Penalty: 1.1 (prevent loops)")
print("   - Streaming Interval: 2.0s (real-time transcription)")
print("   - Rate Limiting: 60 rpm general, 10 rpm inference")
print("   - Inference Queue: max 2 concurrent, 20 queued")
print("   - Monitoring: Prometheus metrics + structured JSON logging")
print("   - Alerts: error rate > 10% warning, > 25% critical")
print("   - HIPAA: Audit trail enabled, encryption at rest configurable")
print("   - Data Retention: 365 days sessions, 2555 days audit logs")

## Step 5: Build the React Frontend (Optional but Recommended)

This step builds the **Clinical Glass** React 19 UI and configures FastAPI to serve it at the root URL.

**What you get over the vanilla JS UI:**
- Glassmorphism dark theme + 5 additional themes (Light, Neon, Midnight, Aurora, High Contrast WCAG AAA)
- SOAP note review workflow — Approve / Edit / Reject per section with full edit history
- Real-time waveform visualizer, live transcript streaming, pipeline progress display
- Floating conversation panel with entity sidebar and emergency detection
- Session history, system monitoring, HIPAA audit, and settings pages
- PWA install prompt + offline support

**Skip this step** to use the original vanilla JS frontend (served at `/` by default).

> **Note:** First run installs Node.js (~1 min) and builds the frontend (~30 sec). Subsequent runs are faster thanks to npm cache.

In [ ]:
import subprocess, os, glob

os.chdir("/content/voice-symptom-triage-assistant")

# Install Node.js 20 LTS if not present
node_check = subprocess.run(["which", "node"], capture_output=True, text=True)
if node_check.returncode != 0:
    print("\u2699\ufe0f Installing Node.js 20 LTS...")
    subprocess.run(["bash", "-c", "curl -fsSL https://deb.nodesource.com/setup_20.x | bash -"],
                   capture_output=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs"], capture_output=True, check=True)
    print(f"\u2705 Node.js installed: {subprocess.check_output(['node', '--version']).decode().strip()}")
else:
    print(f"\u2705 Node.js already installed: {subprocess.check_output(['node', '--version']).decode().strip()}")

# Install frontend dependencies
print("\n\U0001f4e6 Installing frontend dependencies...")
result = subprocess.run(["npm", "ci", "--prefer-offline"], cwd="frontend", capture_output=True, text=True)
if result.returncode != 0:
    # Fallback to npm install if ci fails (no lockfile)
    result = subprocess.run(["npm", "install"], cwd="frontend", capture_output=True, text=True)
    if result.returncode != 0:
        print(f"\u274c npm install failed:\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}")
        raise RuntimeError("npm install failed")

# Build React frontend
print("\U0001f528 Building React frontend...")
result = subprocess.run(["npm", "run", "build"], cwd="frontend", capture_output=True, text=True)
if result.returncode != 0:
    print(f"\u274c Build failed:\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}")
    raise RuntimeError("Vite build failed")

# Verify build output
dist_index = "/content/voice-symptom-triage-assistant/frontend/dist/index.html"
if os.path.exists(dist_index):
    assets = glob.glob("frontend/dist/assets/*")
    total_kb = sum(os.path.getsize(f) for f in assets) // 1024
    print(f"\n\u2705 React frontend built successfully!")
    print(f"   \u2022 {len(assets)} asset files ({total_kb} KB total)")
    print(f"   \u2022 Output: frontend/dist/")
    print(f"\n\U0001f310 The server (Step 6/7) will automatically serve the React UI at the root URL.")
else:
    print("\u274c Build completed but index.html not found. Check build output above.")

## Step 6: Configure ngrok (Required for Public URL)

Get your free authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok
import nest_asyncio

# REPLACE WITH YOUR NGROK AUTHTOKEN
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

ngrok.set_auth_token(NGROK_TOKEN)
nest_asyncio.apply()

print("\u2705 ngrok configured!")

## Step 7: Start the Server

Starts uvicorn with ngrok tunnel. The server auto-detects and serves the React frontend if `frontend/dist` exists.

In [ ]:
import subprocess
import threading
import time
import os
os.environ["MEDGEMMA_TERMS_ACKNOWLEDGED"] = "true"

# Ensure we're in the project directory
os.chdir("/content/voice-symptom-triage-assistant")

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"{'='*60}\n")
print("Open this URL in your browser to access the application!\n")

react_built = os.path.exists("/content/voice-symptom-triage-assistant/frontend/dist/index.html")
print(f"UI: {'\u2705 Clinical Glass React frontend' if react_built else '\u26a0\ufe0f Vanilla JS frontend (run Step 5 to enable React UI)'}")
print(f"CWD: {os.getcwd()}")
print()
print("Features in this deployment:")
print("   - Real-time streaming transcription via WebSocket")
print("   - Live word-by-word transcript display")
print("   - Medical NER \u2014 Condition & Medication extraction (SciSpaCy)")
print("   - Progressive Web App \u2014 Offline recording & background sync")
print("   - API Rate Limiting \u2014 per-user sliding window (60/10 rpm)")
print("   - Inference Queue \u2014 max 2 concurrent, position & ETA tracking")
print("   - Prometheus metrics: GET /metrics")
print("   - Monitoring dashboard: GET /api/monitoring/dashboard (Admin)")
print("   - Active alerts: GET /api/monitoring/alerts (Admin)")
print("   - HIPAA audit trail: GET /api/hipaa/audit-summary (Admin)")
print("   - Data retention stats: GET /api/hipaa/retention/stats (Admin)")
print("   - Export logs: GET /api/hipaa/export-logs (Admin)")
print("   - Structured JSON logging with correlation IDs")
print("   - GPU acceleration for faster results\n")
if react_built:
    print("React UI pages available:")
    print("   /           \u2192 Clinical Glass dashboard")
    print("   /session    \u2192 Voice assistant + SOAP review")
    print("   /history    \u2192 Session history")
    print("   /ambient    \u2192 Ambient documentation mode")
    print("   /monitoring \u2192 System monitoring")
    print("   /hipaa      \u2192 HIPAA compliance & audit trail")
    print("   /settings   \u2192 Themes, audio, model config\n")

# Start uvicorn \u2014 app.main auto-detects and serves React if frontend/dist exists
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

## Step 8: Access Your Application

1. **Copy the public ngrok URL** from the output above
2. **Open it in your browser**
3. **Start recording** — words will appear in real-time as you speak!

### If you ran Step 5 (React UI):
| Route | Page |
|-------|------|
| `/` | Clinical Glass dashboard (stats, recent sessions, system health) |
| `/session` | Voice assistant with waveform, live transcript, and SOAP review |
| `/history` | Searchable session history |
| `/ambient` | Ambient passive documentation mode |
| `/monitoring` | Model performance, queue status, alerts |
| `/hipaa` | HIPAA compliance checklist and audit trail |
| `/settings` | Theme picker (6 themes), audio config, model selection |

### If you skipped Step 5 (Vanilla JS UI):
- Navigate to the root URL and use the original interface
- Click **'Generate Documentation'** to get structured SOAP notes

### To Stop:
- Click the **Stop** button in Colab, or press **Ctrl+C** in the cell output

### Notes:
- Free Colab sessions last **up to 12 hours**
- The ngrok URL **changes each time** you restart
- Models are **cached** after first download (faster restarts)
- GPU inference is **10x faster** than CPU

## Troubleshooting

### If models fail to load:
1. Check your HF_TOKEN is valid
2. Verify you accepted model terms on Hugging Face
3. Try restarting the runtime: Runtime → Restart runtime

### If ngrok fails:
1. Verify your ngrok authtoken
2. Free ngrok accounts have limits (1 tunnel at a time)
3. Try getting a new authtoken from ngrok dashboard

### If audio fails:
1. Check browser microphone permissions
2. Hard refresh browser (Ctrl+Shift+R)
3. Ensure audio files are under 5 minutes (default limit)

### If WebSocket streaming doesn't work:
1. Check browser console (F12) for WebSocket connection errors
2. ngrok tunnels support WebSocket natively — no extra config needed
3. Ensure FFmpeg is available: `!apt-get install -y ffmpeg` (pre-installed on Colab)
4. If streaming fails, the app automatically falls back to upload-based transcription

### If the React UI doesn't load:
1. Verify Step 5 ran successfully: `!ls /content/voice-symptom-triage-assistant/frontend/dist`
2. Check uvicorn startup logs for mount messages
3. If the dist folder is missing, re-run the Step 5 build cell
4. Try a hard refresh in the browser (Ctrl+Shift+R) to clear cached assets
5. Open DevTools → Application → Storage → Clear site data, then refresh

### If documentation shows "N/A" fields:
1. Check the server logs for "parsing_method" messages
2. Look for "json_successful" or "text_extraction_fallback"
3. If seeing frequent fallback, the model may need more warmup time

### If Medical NER shows no entities:
1. Check that SciSpaCy models installed correctly: `import en_ner_bc5cdr_md`
2. Ensure the transcript contains medical terms (conditions or medications)
3. Check server logs for `Medical NER ready: True`

### If you hit rate limits (HTTP 429):
1. The UI shows a red toast with retry countdown — wait for it to expire
2. Default: 60 requests/min general, 10 requests/min for inference endpoints
3. To adjust, update `.env` values: `RATE_LIMIT_GENERAL_RPM`, `RATE_LIMIT_INFERENCE_RPM`

### Debug Mode:
```python
# In a new cell
import logging
logging.getLogger('app.models.medgemma_service').setLevel(logging.DEBUG)
```
Then restart the server to see detailed parsing logs.